In [4]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression, BayesianRidge
from sklearn.neural_network import MLPRegressor
from sklearn.base import BaseEstimator
from typing import Callable
from sklearn.metrics import mean_absolute_error, accuracy_score, mean_absolute_percentage_error, r2_score, root_mean_squared_error
import pandas as pd
import bambi as bmb
import arviz as az
from scipy.stats import norm
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# custom 
from utils import ts_parents, all_formulas_from_structures, train_regime_models_bayesian, classify_regime_bayesian, sliding_window_regime_prediction

In [1]:
def forecasting_metrics(y_test, y_pred):
    return {
        "R-squared (R2)": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
    }


def prepare_train_sets(data, causal_structures, train_n=500):
    train = data[:train_n]
    train_by_regime = {
        regime: train[train["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }
    return {
        regime: ts_parents(train_by_regime[regime], causal_structures[regime])
        for regime in causal_structures
    }


def prepare_classification_sets(data, causal_structures, train_n=500, w=5):
    test = data[train_n - w - 1:]
    true_regime = test["regime"].iloc[w:].to_numpy()
    test_data = test.drop(columns=["regime"])

    X_test = {
        regime: ts_parents(test_data, causal_structures[regime])
        for regime in causal_structures
    }
    return X_test, true_regime


def fit_bayesian_regime_pipeline(data, causal_structures, train_n=500, w=5):
    X_train = prepare_train_sets(data, causal_structures, train_n=train_n)
    all_formulas = all_formulas_from_structures(causal_structures)

    bayesian_models = train_regime_models_bayesian(
        X=X_train,
        all_formulas=all_formulas
    )

    X_test, true_regime = prepare_classification_sets(
        data,
        causal_structures,
        train_n=train_n,
        w=w
    )

    regime_classification, errors = classify_regime_bayesian(
        X_test=X_test,
        all_formulas=all_formulas,
        bayesian_models=bayesian_models,
        causal_structures=causal_structures
    )

    regime_acc_w1 = accuracy_score(regime_classification[w - 1:], true_regime)
    window_preds = sliding_window_regime_prediction(errors, window_size=w)
    regime_acc_w5 = accuracy_score(window_preds, true_regime)

    return bayesian_models, window_preds, regime_acc_w1, regime_acc_w5


def forecast_bayesian_regime_x2(data, causal_structures, bayesian_models, window_preds, train_n=500):
    test_data = data[train_n - 1:].copy()
    test_data.loc[:, "regime"] = window_preds

    test = ts_parents(test_data, causal_structures[0])

    test_features = {
        regime: test[test["regime"] == regime].drop(columns=["regime"])
        for regime in causal_structures
    }

    preds_by_regime = {}
    for regime in causal_structures:
        model = bayesian_models[regime]["X2"]["model"]
        idata = bayesian_models[regime]["X2"]["idata"]

        preds = model.predict(
            idata=idata,
            data=test_features[regime],
            kind="response",
            inplace=False
        ).posterior_predictive["X2"]

        preds_by_regime[regime] = preds.mean(dim=("chain", "draw")).values

    # need to be in correct order 
    y_pred = np.concatenate([preds_by_regime[0], preds_by_regime[1]])
    y_test = test["X2"].values

    return y_test, y_pred


def forecast_markov_switching_x2(data, train_n=500):
    df = data.copy()
    n_vars = len(df.columns)

    for j in range(n_vars):
        df[f"X{j}_lag1"] = df[f"X{j}"].shift(1)

    df = df.dropna().reset_index(drop=True)

    train = df.iloc[:train_n - 1]
    test = df.iloc[train_n - 1:700]

    y_train = train["X2"]
    exog_cols = [f"X{j}_lag1" for j in range(n_vars)]
    exog_train = train[exog_cols]

    model = MarkovRegression(
        endog=y_train,
        k_regimes=2,
        exog=exog_train,
        trend="c",
        switching_trend=True,
        switching_exog=True,
        switching_variance=True
    )
    res = model.fit(em_iter=10, search_reps=5, disp=False)

    p00, p10 = res.params["p[0->0]"], res.params["p[1->0]"]
    P = np.array([[p00, 1 - p00],
                  [p10, 1 - p10]])

    intercepts = np.array([res.params["const[0]"], res.params["const[1]"]])

    betas = np.vstack([
        [res.params[f"x{k+1}[0]"], res.params[f"x{k+1}[1]"]]
        for k in range(n_vars)
    ]).T

    sigmas = np.sqrt([res.params["sigma2[0]"], res.params["sigma2[1]"]])

    pi = res.filtered_marginal_probabilities.iloc[-1].values
    y_pred = []

    for _, row in test.iterrows():
        pi_pred = pi.dot(P)
        lag_vals = np.array([row[c] for c in exog_cols])
        mus = intercepts + (betas * lag_vals).sum(axis=1)
        y_hat = pi_pred.dot(mus)
        y_pred.append(y_hat)

        likelihoods = norm.pdf(row["X2"], loc=mus, scale=sigmas)
        pi = pi_pred * likelihoods
        pi /= pi.sum()

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def forecast_var_x2(data, train_n=500):
    df = data.copy()

    train = df.iloc[:train_n]
    test = df.iloc[train_n:700]

    model = VAR(train)
    res = model.fit(maxlags=1)

    lag_order = res.k_ar
    history = train.values.copy()
    y_pred = []

    for obs in test.values:
        last_obs = history[-lag_order:]
        yhat = res.forecast(y=last_obs, steps=1)
        y_pred.append(yhat[0, 2])
        history = np.vstack([history, obs])

    y_test = test["X2"].values
    return y_test, np.array(y_pred)


def evaluate_simulation(simulation, train_n=500, w=5):
    data = simulation["df"].copy()
    causal_structures = simulation["causal_structure"]

    bayesian_models, window_preds, regime_acc_w1, regime_acc_w5 = fit_bayesian_regime_pipeline(
        data=data,
        causal_structures=causal_structures,
        train_n=train_n,
        w=w
    )

    y_test_bayes, y_pred_bayes = forecast_bayesian_regime_x2(
        data=data,
        causal_structures=causal_structures,
        bayesian_models=bayesian_models,
        window_preds=window_preds,
        train_n=train_n
    )

    y_test_msv, y_pred_msv = forecast_markov_switching_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    y_test_var, y_pred_var = forecast_var_x2(
        data=data.drop(columns=["regime"]),
        train_n=train_n
    )

    # --- regime accuracy rows ---
    regime_rows = [
        {
            "seed": simulation.get("seed"),
            "model": "Bayesian Regime Model",
            "w=1": regime_acc_w1,
            "w=35": regime_acc_w5,
        }
    ]

    # --- forecasting rows ---
    forecast_rows = []

    row_bayes = {
        "seed": simulation.get("seed"),
        "model": "Bayesian Regime Model",
    }
    row_bayes.update(forecasting_metrics(y_test_bayes, y_pred_bayes))
    forecast_rows.append(row_bayes)

    row_msv = {
        "seed": simulation.get("seed"),
        "model": "Markov-Switching VAR",
    }
    row_msv.update(forecasting_metrics(y_test_msv, y_pred_msv))
    forecast_rows.append(row_msv)

    row_var = {
        "seed": simulation.get("seed"),
        "model": "VAR(1)",
    }
    row_var.update(forecasting_metrics(y_test_var, y_pred_var))
    forecast_rows.append(row_var)

    return regime_rows, forecast_rows

In [2]:
import pickle
with open("mc_system2.pkl", "rb") as f:
    causal_discovery_results = pickle.load(f)

len(causal_discovery_results)

5

In [5]:
regime_rows_all = []
forecast_rows_all = []
i = 1
for simulation in causal_discovery_results:
    print(f"forecasting monte carlo {i}...")
    regime_rows, forecast_rows = evaluate_simulation(simulation, train_n=500, w=35)
    regime_rows_all.extend(regime_rows)
    forecast_rows_all.extend(forecast_rows)
    i += 1

regime_df = pd.DataFrame(regime_rows_all)
forecast_df = pd.DataFrame(forecast_rows_all)

forecasting monte carlo 1...
forecasting monte carlo 2...
forecasting monte carlo 3...
forecasting monte carlo 4...
forecasting monte carlo 5...


## Forecasting Results

In [6]:
# full results 
forecast_df

,seed,model,R-squared (R2),MAE,RMSE,MAPE
0,29,Bayesian Regime Model,0.249960,0.795257,1.034093,2.917509
1,29,Markov-Switching VAR,0.211470,0.807645,1.060294,2.403909
2,29,VAR(1),0.210941,0.806803,1.060650,2.389409
3,34,Bayesian Regime Model,0.400782,0.780586,0.991795,2.241054
4,34,Markov-Switching VAR,0.373857,0.799566,1.013833,2.053128
5,34,VAR(1),0.353371,0.819006,1.030285,1.724861
6,49,Bayesian Regime Model,0.404808,0.769950,0.974436,7.161350
7,49,Markov-Switching VAR,0.314256,0.849915,1.045937,5.628939
8,49,VAR(1),0.314933,0.849589,1.045421,5.690352
9,105,Bayesian Regime Model,0.283148,0.793734,1.005834,3.707135


In [8]:
# test set regime classification
regime_df[["w=1","w=35"]].agg(["mean", "std"])

,w=1,w=35
mean,0.637811,0.930348
std,0.029517,0.012684


In [9]:
# summary with mean and standard deviation over monte carlo simulations
forecast_summary = forecast_df.drop(columns=["seed"]).groupby("model").agg(["mean", "std"])
forecast_summary

R-squared (R2)                 MAE                RMSE  \
                                mean       std      mean       std      mean   
model                                                                          
Bayesian Regime Model       0.330923  0.069645  0.787092  0.011458  0.995936   
Markov-Switching VAR        0.291106  0.059353  0.812011  0.021421  1.025930   
VAR(1)                      0.285255  0.053227  0.818566  0.018130  1.030428   

                                     MAPE            
                            std      mean       std  
model                                                
Bayesian Regime Model  0.025178  3.510727  2.194677  
Markov-Switching VAR   0.025655  2.858161  1.643478  
VAR(1)                 0.023291  3.041727  1.800839

In [10]:
forecast_summary.to_latex()

'\\begin{tabular}{lrrrrrrrr}\n\\toprule\n & \\multicolumn{2}{r}{R-squared (R2)} & \\multicolumn{2}{r}{MAE} & \\multicolumn{2}{r}{RMSE} & \\multicolumn{2}{r}{MAPE} \\\\\n & mean & std & mean & std & mean & std & mean & std \\\\\nmodel &  &  &  &  &  &  &  &  \\\\\n\\midrule\nBayesian Regime Model & 0.330923 & 0.069645 & 0.787092 & 0.011458 & 0.995936 & 0.025178 & 3.510727 & 2.194677 \\\\\nMarkov-Switching VAR & 0.291106 & 0.059353 & 0.812011 & 0.021421 & 1.025930 & 0.025655 & 2.858161 & 1.643478 \\\\\nVAR(1) & 0.285255 & 0.053227 & 0.818566 & 0.018130 & 1.030428 & 0.023291 & 3.041727 & 1.800839 \\\\\n\\bottomrule\n\\end{tabular}\n'